In [0]:
gold_df = spark.read.table(
    "finops_catalog_2.finops.gold_anomaly_features"
)

pred_df = spark.read.table(
    "finops_catalog_2.finops.ml_predictions"
)

dashboard_df = gold_df.join(
    pred_df.select(
        "date",
        "net_cost",
        "total_savings",
        "pred_anomaly"
    ),
    ["date", "net_cost"],
    "left"
)
dashboard_df.printSchema()


In [0]:
from pyspark.sql.functions import col, round, when

dashboard_df = (
    dashboard_df
    .withColumn("net_cost", round(col("net_cost"), 2))
    .withColumn("budget_amount", round(col("budget_amount"), 2))
    .withColumn("total_savings", round(col("total_savings"), 2))
    .withColumn("anomaly_score", round(col("anomaly_score"), 2))
    .withColumn("usage_quantity", round(col("usage_quantity"), 2))
    .withColumn("budget_utilization_pct", round(col("budget_utilization_pct"), 4))
    .withColumn("anomaly_status", when(col("pred_anomaly") == 1, "Anomaly").otherwise("Normal"))
)
display(dashboard_df.limit(5))

In [0]:
dashboard_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "finops_catalog_2.finops.dashboard_features"
    )
display(dashboard_df.limit(5))

In [0]:
%sql
SELECT anomaly_status, COUNT(*)
FROM finops_catalog_2.finops.dashboard_features
GROUP BY anomaly_status;